# Train B1 — YOLOv8n-P2 (100 epochs) — Kaggle GPU
**KLTN: PCB Defect Detection**

### Cấu trúc dataset Kaggle đã mount:
```
/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/
    PKU-Market-PCB(Data enhanced version)/
        train/images/*.jpg  + train/labels/*.txt
        valid/images/*.jpg  + valid/labels/*.txt
        test/images/*.jpg   + test/labels/*.txt

/kaggle/input/datasets/tanthongthai/   ← project code (configs/ + data/splits/)
```

### ⚠️ Điểm khác biệt quan trọng so với B0:
- Model B1 dùng kiến trúc **`configs/yolov8n-p2.yaml`** (thêm tầng P2 stride-4) thay vì `yolov8n.pt`.
- Khởi tạo bằng cách: `YOLO("yolov8n-p2.yaml").load("yolov8n.pt")` → load pretrained backbone nhưng dùng P2 head.
- Output zip: `B1_100ep_weights.zip`

### Cách chạy:
1. Add cả 2 dataset vào notebook
2. Chọn **GPU T4 x2** + **Internet ON**
3. **Run All**


In [1]:
# Fix Ray Tune conflict (giống hệt B0)
!pip uninstall ray -y -q 2>/dev/null || True
!pip install ultralytics==8.3.40 -q --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 14.2 MB/s eta 0:00:00a 0:00:01


In [2]:
import os, json, shutil, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

import yaml
from ultralytics import YOLO

# ── Kaggle dataset paths ─────────────────────────────────────────────────────
KAGGLE_IMG_ROOT = Path(
    "/kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset"
    "/PKU-Market-PCB(Data enhanced version)"
)

# Project code dataset (tanthongthai) — chứa configs/ và data/splits/
KAGGLE_CODE_ROOT = Path("/kaggle/input/datasets/tanthongthai/kltn-pcb-code1")

WORK_DIR = Path("/kaggle/working")

# ── Verify image dataset ─────────────────────────────────────────────────────
assert KAGGLE_IMG_ROOT.exists(), f"❌ Không tìm thấy dataset ảnh: {KAGGLE_IMG_ROOT}"

print("✅ Dataset ảnh:", KAGGLE_IMG_ROOT)
print("   Train images:", len(list((KAGGLE_IMG_ROOT/"train"/"images").glob("*.jpg"))))
print("   Valid images:", len(list((KAGGLE_IMG_ROOT/"valid"/"images").glob("*.jpg"))))
print("   Test images :", len(list((KAGGLE_IMG_ROOT/"test" /"images").glob("*.jpg"))))
print()

# ── Tìm CODE_ROOT (chứa configs/base.yaml) ──────────────────────────────────
CODE_ROOT = None
for candidate in [
    KAGGLE_CODE_ROOT,
    *KAGGLE_CODE_ROOT.glob("*/"),
    *KAGGLE_CODE_ROOT.glob("*/*/"),
]:
    if (candidate / "configs" / "base.yaml").exists():
        CODE_ROOT = candidate
        break

if CODE_ROOT:
    print(f"✅ Project code: {CODE_ROOT}")
    print(f"   yolov8n-p2.yaml exists: {(CODE_ROOT / 'configs' / 'yolov8n-p2.yaml').exists()}")
else:
    print("⚠️  Không tìm thấy configs/base.yaml trong tanthongthai dataset")
    print("   Sẽ dùng hyperparams mặc định.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Dataset ảnh: /kaggle/input/datasets/liuxiaolong1/pcb-defect-detection-dataset/PKU-Market-PCB(Data enhanced version)
   Train images: 2772
   Valid images: 366
   Test images : 366

✅ Project code: /kaggle/input/datasets/tanthongthai/kltn-pcb-code1
   yolov8n-p2.yaml exists: True


In [3]:
# ── Load từ base.yaml nếu có, fallback về default ───────────────────────────
if CODE_ROOT and (CODE_ROOT / "configs" / "base.yaml").exists():
    with open(CODE_ROOT / "configs" / "base.yaml") as f:
        cfg = yaml.safe_load(f)
    print("✅ Loaded configs/base.yaml:", cfg)
else:
    cfg = {"batch": 16, "imgsz": 640, "seed": 41, "patience": 30}
    print("⚠️  Dùng hyperparams mặc định:", cfg)

# ===== CHỈ OVERRIDE EPOCHS — mọi tham số khác từ base.yaml =====
EPOCHS   = 100
BATCH    = cfg.get("batch",    16)
IMGSZ    = cfg.get("imgsz",   640)
SEED     = cfg.get("seed",     41)
PATIENCE = cfg.get("patience", 30)
# ================================================================

print(f"\nHyperparams sẽ dùng:")
print(f"  epochs  : {EPOCHS}  (override — so sánh công bằng với B0_100ep)")
print(f"  batch   : {BATCH}")
print(f"  imgsz   : {IMGSZ}")
print(f"  seed    : {SEED}")
print(f"  patience: {PATIENCE}")


✅ Loaded configs/base.yaml: {'model': 'yolov8n.pt', 'imgsz': 640, 'epochs': 20, 'batch': 16, 'optimizer': 'auto', 'seed': 41, 'patience': 30, 'classes': ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper'], 'split_ratio': [0.7, 0.15, 0.15]}

Hyperparams sẽ dùng:
  epochs  : 100  (override — so sánh công bằng với B0_100ep)
  batch   : 16
  imgsz   : 640
  seed    : 41
  patience: 30


In [4]:
# ── Load split files từ project code ────────────────────────────────────────

LOCAL_PREFIX = "data/raw/PKU-Market-PCB(Data enhanced version)/"

def remap_split(txt_path: Path, kaggle_root: Path) -> list:
    """
    Đọc split .txt, remap relative path → Kaggle absolute path.

    Local  : data/raw/PKU-Market-PCB(Data enhanced version)/train/images/xx.jpg
    Kaggle : /kaggle/input/.../PKU-Market-PCB(Data enhanced version)/train/images/xx.jpg
    """
    lines = txt_path.read_text().strip().splitlines()
    remapped = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith(LOCAL_PREFIX):
            rel = line[len(LOCAL_PREFIX):]  # e.g. "train/images/xx.jpg"
            new_path = str(kaggle_root / rel)
        else:
            new_path = str(kaggle_root / line)
        remapped.append(new_path)
    return remapped

splits_loaded = False

if CODE_ROOT:
    train_txt = CODE_ROOT / "data" / "splits" / "train.txt"
    val_txt   = CODE_ROOT / "data" / "splits" / "val.txt"

    if train_txt.exists() and val_txt.exists():
        train_paths = remap_split(train_txt, KAGGLE_IMG_ROOT)
        val_paths   = remap_split(val_txt,   KAGGLE_IMG_ROOT)
        splits_loaded = True
        print(f"✅ Loaded splits từ project code: train={len(train_paths)}, val={len(val_paths)}")

if not splits_loaded:
    print("⚠️  Không load được split → scan toàn bộ thư mục ảnh")
    import random
    random.seed(SEED)

    all_imgs = sorted((KAGGLE_IMG_ROOT / "train" / "images").glob("*.jpg"))
    all_imgs += sorted((KAGGLE_IMG_ROOT / "valid" / "images").glob("*.jpg"))
    all_imgs = [str(p) for p in all_imgs]

    random.shuffle(all_imgs)
    n_val = int(len(all_imgs) * 0.15)
    val_paths   = all_imgs[:n_val]
    train_paths = all_imgs[n_val:]
    splits_loaded = True
    print(f"   Scan fallback: train={len(train_paths)}, val={len(val_paths)}")

def check_paths(paths, name, n=5):
    missing = [p for p in paths[:n] if not Path(p).exists()]
    if missing:
        print(f"⚠️  {name}: {len(missing)}/{n} sample paths KHÔNG tồn tại:")
        for p in missing:
            print(f"   {p}")
        return False
    print(f"✅ {name}: {n} sample paths tồn tại")
    return True

ok_train = check_paths(train_paths, "train")
ok_val   = check_paths(val_paths,   "val")

if not (ok_train and ok_val):
    raise RuntimeError("Path remapping thất bại. Kiểm tra lại KAGGLE_IMG_ROOT.")


✅ Loaded splits từ project code: train=3989, val=368
✅ train: 5 sample paths tồn tại
✅ val: 5 sample paths tồn tại


In [5]:
# ── Ghi split files vào working dir ─────────────────────────────────────────
WORK_DIR.mkdir(parents=True, exist_ok=True)
splits_dir = WORK_DIR / "splits"
splits_dir.mkdir(exist_ok=True)

train_txt_kaggle = splits_dir / "train.txt"
val_txt_kaggle   = splits_dir / "val.txt"

train_txt_kaggle.write_text("\n".join(train_paths) + "\n")
val_txt_kaggle.write_text(  "\n".join(val_paths)   + "\n")

print(f"✅ Wrote train.txt: {len(train_paths)} images")
print(f"✅ Wrote val.txt  : {len(val_paths)} images")

# ── Tạo data.yaml cho Kaggle ──────────────────────────────────────────────────
data_yaml = {
    "path"  : str(WORK_DIR),
    "train" : str(train_txt_kaggle),
    "val"   : str(val_txt_kaggle),
    # test KHÔNG được thêm để tránh data leakage
    "nc"    : 6,
    "names" : {
        0: "missing_hole",
        1: "mouse_bite",
        2: "open_circuit",
        3: "short",
        4: "spur",
        5: "spurious_copper",
    },
}

data_yaml_path = WORK_DIR / "data_kaggle.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ data_kaggle.yaml:")
with open(data_yaml_path) as f:
    print(f.read())


✅ Wrote train.txt: 3989 images
✅ Wrote val.txt  : 368 images

✅ data_kaggle.yaml:
names:
  0: missing_hole
  1: mouse_bite
  2: open_circuit
  3: short
  4: spur
  5: spurious_copper
nc: 6
path: /kaggle/working
train: /kaggle/working/splits/train.txt
val: /kaggle/working/splits/val.txt



In [6]:
# ── Copy yolov8n-p2.yaml vào working dir ────────────────────────────────────
# Ultralytics cần đọc được file yaml từ writable path trên Kaggle

P2_YAML_SRC = CODE_ROOT / "configs" / "yolov8n-p2.yaml"
P2_YAML_DST = WORK_DIR / "yolov8n-p2.yaml"

assert P2_YAML_SRC.exists(), f"❌ Không tìm thấy {P2_YAML_SRC}"
shutil.copy(P2_YAML_SRC, P2_YAML_DST)
print(f"✅ Copied yolov8n-p2.yaml → {P2_YAML_DST}")

# Verify nội dung file
with open(P2_YAML_DST) as f:
    p2_content = f.read()

print(f"\n--- Nội dung yolov8n-p2.yaml (30 dòng đầu) ---")
for line in p2_content.splitlines()[:30]:
    print(line)


✅ Copied yolov8n-p2.yaml → /kaggle/working/yolov8n-p2.yaml

--- Nội dung yolov8n-p2.yaml (30 dòng đầu) ---
# Ultralytics YOLOv8n-P2 configuration for PCB Defect Detection (KLTN PCB - Week 7-8)
# Architecture: High-resolution P2 layer added for tiny defect detection
# Feature map outputs: P2 (stride 4, 160x160) - P5 (stride 32, 20x20) at input 640x640

# Parameters
nc: 6 # 6 PCB defect classes: missing_hole, mouse_bite, open_circuit, short, spur, spurious_copper
scales: # model compound scaling constants for 'n' (YOLOv8n)
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]

# YOLOv8 backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]    # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]   # 1-P2/4
  - [-1, 3, C2f, [128, True]]    # 2
  - [-1, 1, Conv, [256, 3, 2]]   # 3-P3/8
  - [-1, 6, C2f, [256, True]]    # 4
  - [-1, 1, Conv, [512, 3, 2]]   # 5-P4/16
  - [-1, 6, C2f, [512, True]]    # 6
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]

In [7]:
# Fix Ray Tune incompatibility — PHẢI chạy trước model.train()

# Fix 1: patch ray session API bị missing
try:
    import ray.train._internal.session as _ray_session
    if not hasattr(_ray_session, '_get_session'):
        _ray_session._get_session = lambda: None
        print("✅ Patched ray._get_session")
except Exception as e:
    print(f"ray patch skipped: {e}")

# Fix 2: vô hiệu hóa thẳng Ultralytics raytune callback
try:
    import ultralytics.utils.callbacks.raytune as _raytune_cb
    _raytune_cb.on_fit_epoch_end = lambda trainer: None
    print("✅ Disabled ultralytics raytune callback")
except Exception as e:
    print(f"raytune patch skipped: {e}")


ray patch skipped: No module named 'ray'
✅ Disabled ultralytics raytune callback


In [8]:
RUN_NAME = "B1_100ep"
t0 = time.time()

# ── Khởi tạo model B1 (P2 architecture + pretrained backbone) ────────────────
# Dùng YOLO(yaml).load(weights) thay vì YOLO(weights) để load đúng P2 head
print(f"Khởi tạo B1 từ {P2_YAML_DST.name} + load pretrained backbone yolov8n.pt ...")
model = YOLO(str(P2_YAML_DST)).load("yolov8n.pt")

print(f"\n=== BẮT ĐẦU HUẤN LUYỆN B1 ({EPOCHS} EPOCHS, BATCH {BATCH}) ===")
results = model.train(
    data     = str(data_yaml_path),
    epochs   = EPOCHS,
    batch    = BATCH,
    imgsz    = IMGSZ,
    seed     = SEED,
    patience = PATIENCE,
    project  = str(WORK_DIR / "runs"),
    name     = RUN_NAME,
    device   = 0,        # GPU Kaggle (T4)
    verbose  = True,
    save     = True,
    plots    = True,
    exist_ok = True,
    workers  = 2,
    # Augmentation recipe đã khóa cho PCB
    degrees     = 0.0,
    shear       = 0.0,
    perspective = 0.0,
    flipud      = 0.0,
    fliplr      = 0.5,
    mosaic      = 1.0,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    scale       = 0.5,
    translate   = 0.1,
)

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Training hoàn tất! Elapsed: {elapsed/60:.1f} phút")
print(f"{'='*60}")

m = results.results_dict
print(f"mAP50    : {m.get('metrics/mAP50(B)',    0):.4f}")
print(f"mAP50-95 : {m.get('metrics/mAP50-95(B)',0):.4f}")
print(f"Precision: {m.get('metrics/precision(B)',0):.4f}")
print(f"Recall   : {m.get('metrics/recall(B)',   0):.4f}")


Khởi tạo B1 từ yolov8n-p2.yaml + load pretrained backbone yolov8n.pt ...


100%|██████████| 6.25M/6.25M [00:00<00:00, 80.8MB/s]

Transferred 219/437 items from pretrained weights



=== BẮT ĐẦU HUẤN LUYỆN B1 (100 EPOCHS, BATCH 16) ===
New https://pypi.org/project/ultralytics/8.4.162 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/working/yolov8n-p2.yaml, data=/kaggle/working/data_kaggle.yaml, epochs=100, time=None, patience=30, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=/kaggle/working/runs, name=B1_100ep, exist_ok=True, pretrained=yolov8n.pt, optimizer=auto, verbose=True, seed=41, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, 

100%|██████████| 755k/755k [00:00<00:00, 17.1MB/s]


...

     64/100      3.89G      1.267      1.007      1.028          9        640: 100%|██████████| 250/250 [02:16<00:00,  1.83it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:04<00:00,  2.45it/s]

                   all        368       1860      0.739      0.654       0.64      0.215


EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 34, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

64 epochs completed in 2.628 hours.
Optimizer stripped from /kaggle/working/runs/B1_100ep/weights/last.pt, 6.2MB
Optimizer stripped from /kaggle/working/runs/B1_100ep/weights/best.pt, 6.2MB

Validating /kaggle/working/runs/B1_100ep/weights/best.pt...
WARNING ⚠️ validating an untrained model YAML will result in 0 mAP.
Ultralytics 8.3.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n-p2 summary (fused): 207 layers, 2,921,832 parameters, 0 gradients, 12.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 12/12 [00:06<00:00,  1.90it/s]


                   all        368       1860      0.709      0.681      0.715      0.296
          missing_hole         60        300      0.243      0.917      0.795      0.227
            mouse_bite         60        304       0.99       0.64      0.774      0.373
          open_circuit         64        324      0.679      0.647      0.617      0.304
                 short         60        304      0.872      0.707      0.797      0.331
                  spur         60        300      0.554       0.54      0.523      0.228
       spurious_copper         64        328      0.916      0.635      0.782       0.31
Speed: 0.2ms preprocess, 3.5ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /kaggle/working/runs/B1_100ep

Training hoàn tất! Elapsed: 158.9 phút
mAP50    : 0.7147
mAP50-95 : 0.2955
Precision: 0.7091
Recall   : 0.6809


In [9]:
RUN_DIR = WORK_DIR / "runs" / RUN_NAME

# GPU info
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used",
     "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

m = results.results_dict
meta = {
    "model"       : "yolov8n-p2.yaml (pretrained backbone: yolov8n.pt)",
    "epochs"      : EPOCHS,
    "batch"       : BATCH,
    "imgsz"       : IMGSZ,
    "seed"        : SEED,
    "patience"    : PATIENCE,
    "environment" : "kaggle-gpu",
    "gpu"         : gpu_info,
    "trained_at"  : datetime.now(timezone.utc).isoformat(),
    "elapsed_min" : round(elapsed / 60, 2),
    "train_imgs"  : len(train_paths),
    "val_imgs"    : len(val_paths),
    "map50"       : m.get("metrics/mAP50(B)"),
    "map50_95"    : m.get("metrics/mAP50-95(B)"),
    "precision"   : m.get("metrics/precision(B)"),
    "recall"      : m.get("metrics/recall(B)"),
    "split_source": "project_splits" if splits_loaded and CODE_ROOT else "scan_fallback",
    "architecture": "YOLOv8n-P2 (4 detection heads: P2/4, P3/8, P4/16, P5/32)",
}

with open(RUN_DIR / "run_metadata.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

with open(RUN_DIR / "gpu_info.txt", "w") as f:
    f.write(gpu_info + "\n")

print("✅ Metadata saved:")
print(json.dumps(meta, indent=2, ensure_ascii=False))


✅ Metadata saved:
{
  "model": "yolov8n-p2.yaml (pretrained backbone: yolov8n.pt)",
  "epochs": 100,
  "batch": 16,
  "imgsz": 640,
  "seed": 41,
  "patience": 30,
  "environment": "kaggle-gpu",
  "gpu": "Tesla T4, 15360 MiB, 415 MiB\nTesla T4, 15360 MiB, 0 MiB",
  "trained_at": "2026-09-25T05:04:19.595834+00:00",
  "elapsed_min": 158.86,
  "train_imgs": 3989,
  "val_imgs": 368,
  "map50": 0.7146708336639914,
  "map50_95": 0.29552132469393466,
  "precision": 0.7090673330721907,
  "recall": 0.6809235564138679,
  "split_source": "project_splits",
  "architecture": "YOLOv8n-P2 (4 detection heads: P2/4, P3/8, P4/16, P5/32)"
}


In [11]:
OUT_ZIP = WORK_DIR / "B1_100ep_weights.zip"
INCLUDE_EXT = {".pt", ".csv", ".yaml", ".json", ".png", ".txt"}

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in RUN_DIR.rglob("*"):
        if f.is_file() and f.suffix in INCLUDE_EXT:
            zf.write(f, f.relative_to(WORK_DIR))

size_mb = OUT_ZIP.stat().st_size / 1e6
print(f"✅ Packed: B1_100ep_weights.zip  ({size_mb:.1f} MB)")
print("\nNội dung zip:")
with zipfile.ZipFile(OUT_ZIP) as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"  {name:55s}  {info.file_size/1024:7.1f} KB")
print("\n👉 Download file này từ Kaggle Output tab")


✅ Packed: B1_100ep_weights.zip  (12.5 MB)

Nội dung zip:
  runs/B1_100ep/F1_curve.png                                 272.0 KB
  runs/B1_100ep/PR_curve.png                                 227.2 KB
  runs/B1_100ep/P_curve.png                                  230.0 KB
  runs/B1_100ep/R_curve.png                                  244.2 KB
  runs/B1_100ep/args.yaml                                      1.6 KB
  runs/B1_100ep/confusion_matrix.png                         148.3 KB
  runs/B1_100ep/confusion_matrix_normalized.png              155.4 KB
  runs/B1_100ep/gpu_info.txt                                   0.1 KB
  runs/B1_100ep/results.csv                                    7.7 KB
  runs/B1_100ep/results.png                                  337.6 KB
  runs/B1_100ep/run_metadata.json                              0.6 KB
  runs/B1_100ep/weights/best.pt                             6075.3 KB
  runs/B1_100ep/weights/last.pt                             6075.3 KB

👉 Download file này từ Kaggle Ou